# Final Tract-Level Merge

I'm combining the cleaned tract-level datasets into one master file.

The goal is one row per San Diego County census tract.

ACS is the base dataset because it has the full tract list. From there, I'll merge the other datasets using a cleaned `tract_id` column.

I’ll keep the full master file for now, and try to avoid dupe columns. Once everything is merged, I'll make a smaller scoring dataset with only the features I actually need.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', 150)
pd.set_option('display.max_rows', 100)

# local project path
project_path = Path.cwd().parent

data_path = project_path / 'data'
processed_path = data_path / 'processed'

paths = {
    'acs': processed_path / 'acs_tracts_selected_2024.csv',
    'crime': processed_path / 'crime_safety_by_tract.csv',
    'walkability': processed_path / 'walkability_by_tract.csv',
    'transit': processed_path / 'transit_access_by_tract.csv',
    'environment': processed_path / 'environmental_burden_by_tract.csv',
    'schools': processed_path / 'school_features_by_tract.csv',
    'output': processed_path / 'master_tract_features.csv'
}

In [2]:
# loading each cleaned tract-level dataset
acs = pd.read_csv(paths['acs'])
crime = pd.read_csv(paths['crime'])
walkability = pd.read_csv(paths['walkability'])
transit = pd.read_csv(paths['transit'])
environment = pd.read_csv(paths['environment'])
schools = pd.read_csv(paths['schools'])

datasets = {
    'acs': acs,
    'crime': crime,
    'walkability': walkability,
    'transit': transit,
    'environment': environment,
    'schools': schools}

In [3]:
# checking rows and columns before merging
for name, df in datasets.items():
    print(f'{name}: {df.shape[0]} rows, {df.shape[1]} columns')

acs: 737 rows, 50 columns
crime: 737 rows, 13 columns
walkability: 628 rows, 16 columns
transit: 737 rows, 7 columns
environment: 736 rows, 109 columns
schools: 737 rows, 27 columns


In [4]:
# looking for tract ID columns in each dataset
for name, df in datasets.items():
    tract_cols = [
        col for col in df.columns
        if 'tract' in col.lower() or 'geoid' in col.lower() or 'geo_id' in col.lower()]

    print(f'\n{name.upper()}')
    print(tract_cols)


ACS
['geo_id', 'tract_name', 'tract_type_flag']

CRIME
['geo_id', 'tract_name']

WALKABILITY
['tract_geoid']

TRANSIT
['GEOID', 'tract_area_sq_mile', 'large_tract_flag']

ENVIRONMENT
['census_tract', 'census_tract_fips_code', 'tract_fips']

SCHOOLS
['GEOID', 'tract_area_sq_mile']


In [5]:
# checking possible tract ID values and data types
id_checks = {
    'acs': 'geo_id',
    'crime': 'geo_id',
    'walkability': 'tract_geoid',
    'transit': 'GEOID',
    'environment_census_tract': 'census_tract',
    'environment_fips_code': 'census_tract_fips_code',
    'environment_tract_fips': 'tract_fips',
    'schools': 'GEOID'}

for name, col in id_checks.items():
    if name.startswith('environment'):
        df = environment
    else:
        df_name = name
        df = datasets[df_name]

    print(f'\n{name.upper()} - {col}')
    print('dtype:', df[col].dtype)
    print(df[col].head(10).tolist())



ACS - geo_id
dtype: str
['1400000US06073000100', '1400000US06073000201', '1400000US06073000202', '1400000US06073000301', '1400000US06073000302', '1400000US06073000400', '1400000US06073000500', '1400000US06073000600', '1400000US06073000700', '1400000US06073000800']

CRIME - geo_id
dtype: int64
[6073000100, 6073000201, 6073000202, 6073000301, 6073000302, 6073000400, 6073000500, 6073000600, 6073000700, 6073000800]

WALKABILITY - tract_geoid
dtype: int64
[6073000100, 6073000201, 6073000202, 6073000300, 6073000400, 6073000500, 6073000600, 6073000700, 6073000800, 6073000900]

TRANSIT - GEOID
dtype: int64
[6073008331, 6073008336, 6073008337, 6073011601, 6073011602, 6073011700, 6073011801, 6073013307, 6073013308, 6073013309]

ENVIRONMENT_CENSUS_TRACT - census_tract
dtype: int64
[100, 201, 202, 301, 302, 400, 500, 600, 700, 800]

ENVIRONMENT_FIPS_CODE - census_tract_fips_code
dtype: int64
[6073000100, 6073000201, 6073000202, 6073000301, 6073000302, 6073000400, 6073000500, 6073000600, 607300070

In [6]:
# creating one tract ID column for each dataset
acs['tract_id'] = acs['geo_id'].astype(str).str.replace('1400000US', '', regex=False)

crime['tract_id'] = crime['geo_id'].astype(str).str.zfill(11)
walkability['tract_id'] = walkability['tract_geoid'].astype(str).str.zfill(11)
transit['tract_id'] = transit['GEOID'].astype(str).str.zfill(11)
environment['tract_id'] = environment['census_tract_fips_code'].astype(str).str.zfill(11)
schools['tract_id'] = schools['GEOID'].astype(str).str.zfill(11)

In [7]:
# checking tract ID format in each dataset
for name, df in datasets.items():
    tract_id_lengths = df['tract_id'].str.len().unique()
    tract_count = df['tract_id'].nunique()
    row_count = len(df)

    print(f'{name.upper()}')
    print(f'tract id length: {tract_id_lengths}')
    print(f'number of rows: {row_count}')
    print()

ACS
tract id length: [11]
number of rows: 737

CRIME
tract id length: [11]
number of rows: 737

WALKABILITY
tract id length: [11]
number of rows: 628

TRANSIT
tract id length: [11]
number of rows: 737

ENVIRONMENT
tract id length: [11]
number of rows: 736

SCHOOLS
tract id length: [11]
number of rows: 737



Going to start merging by setting ACS as the "master" table for the tracts, the going to merge on tract ID.

In [8]:
# starting with ACS as the base table
master = acs.copy()

print(f'master rows before merge: {master.shape[0]}')
print(f'master columns before merge: {master.shape[1]}')


master rows before merge: 737
master columns before merge: 51


In [9]:
# merging crime features onto ACS base
master = master.merge(
    crime.drop(columns=['geo_id', 'tract_name'], errors='ignore'),
    on='tract_id',
    how='left')

print(f'master rows after crime merge: {master.shape[0]}')
print(f'master columns after crime merge: {master.shape[1]}')


master rows after crime merge: 737
master columns after crime merge: 62


In [10]:
# merging walkability features
master = master.merge(
    walkability.drop(columns=['tract_geoid'], errors='ignore'),
    on='tract_id',
    how='left')

print(f'master rows after walkability merge: {master.shape[0]}')
print(f'master columns after walkability merge: {master.shape[1]}')

master rows after walkability merge: 737
master columns after walkability merge: 77


In [11]:
# merging transit features
master = master.merge(
    transit.drop(columns=['GEOID'], errors='ignore'),
    on='tract_id',
    how='left')

print(f'master rows after transit merge: {master.shape[0]}')
print(f'master columns after transit merge: {master.shape[1]}')

master rows after transit merge: 737
master columns after transit merge: 83


In [12]:
# merging environmental burden features
master = master.merge(
    environment.drop(columns=['census_tract', 'census_tract_fips_code', 'tract_fips'], errors='ignore'),
    on='tract_id',
    how='left')

print(f'master rows after environment merge: {master.shape[0]}')
print(f'master columns after environment merge: {master.shape[1]}')

master rows after environment merge: 737
master columns after environment merge: 189


In [13]:
# checking school columns before merging
schools.columns.tolist()

['GEOID',
 'NAMELSAD',
 'school_count',
 'elementary_school_count',
 'middle_school_count',
 'high_school_count',
 'other_school_count',
 'charter_school_count',
 'tract_area_sq_mile',
 'schools_per_sq_mile',
 'district_name',
 'district_type',
 'district_grade_low',
 'district_grade_high',
 'district_overlap_pct',
 'ela_students_tested',
 'ela_current_status',
 'ela_status_level',
 'ela_change',
 'ela_change_level',
 'math_students_tested',
 'math_current_status',
 'math_status_level',
 'math_change',
 'math_change_level',
 'academic_strength_score',
 'academic_strength_tier',
 'tract_id']

In [14]:
# checking final row and column count
print(f'master rows: {master.shape[0]}')
print(f'master columns: {master.shape[1]}')

master rows: 737
master columns: 189


In [15]:
# merging school features
master = master.merge(
    schools.drop(columns=['GEOID'], errors='ignore'),
    on='tract_id',
    how='left')

print(f'master rows after schools merge: {master.shape[0]}')
print(f'master columns after schools merge: {master.shape[1]}')

master rows after schools merge: 737
master columns after schools merge: 215


In [16]:
# checking final row count and duplicate tract IDs
print(f'master rows: {master.shape[0]}')
print(f'master columns: {master.shape[1]}')
print(f'duplicated tract IDs: {master["tract_id"].duplicated().sum()}')
print(f'unique tracts: {master["tract_id"].nunique()}')

master rows: 737
master columns: 215
duplicated tract IDs: 0
unique tracts: 737


In [17]:
master.shape

(737, 215)

## Check missing values

The merge worked, so now I’ll check missing values.

There are some missing values, like walkability has fewer tracts than ACS, so some walkability fields might be blank. I don’t want to fill anything automatically until I'm clear on what’s missing.

In [18]:
# checking columns with missing values
missing_summary = (
    master.isna()
    .sum() # get a sum of all the missing values
    .reset_index()
    .rename(columns={'index': 'column', 0: 'missing_count'}))

missing_summary['missing_pct'] = (
    missing_summary['missing_count'] / len(master) * 100).round(2)

missing_summary = missing_summary[missing_summary['missing_count'] > 0]

missing_summary.sort_values('missing_count', ascending=False).head(50)

,column,missing_count,missing_pct
176,tsunami_hazard_type_risk_index_score,653,88.60
169,tsunami_exposure_population,653,88.60
175,tsunami_expected_annual_loss_rate_national_per...,653,88.60
168,tsunami_exposure_building_value,653,88.60
167,tsunami_annualized_frequency,653,88.60
171,tsunami_expected_annual_loss_total,653,88.60
172,tsunami_expected_annual_loss_score,653,88.60
170,tsunami_expected_annual_loss_building_value,653,88.60
174,tsunami_expected_annual_loss_rate_building,653,88.60
67,walk_land_acres,201,27.27


After looking into this more closely, I've decided that 700+ tracts is a lot to look at individually, and the scope for looking at various investment opportunities could be too broad.

I want to narrow down the project to look at development projects that are suitable for families. After suburban sprawl, the rising trends were that people would flock to cities and there would be more single people, therefore, development in the 2000s were focused on urban high-rises suitable for 1-2 people, amenities, and proximity to work. However, there have been more recent discussions that those options are no longer serving the people who are having families later.

My goal new goal is to find safe neighborhoods, with access to schools, where there's an opportunity for family development.

## Preview merged dataset

The merge is done, but I’m not filling missing values yet.

First I want to preview the merged table, check the shape, and save the full merged version as processed data.

In [19]:
# previewing the merged dataset
print(f'master rows: {master.shape[0]}')
print(f'master columns: {master.shape[1]}')

master.head()

master rows: 737
master columns: 215


,geo_id,tract_name,total_population_x,population_under_5,population_under_5_rate,population_under_18,population_under_18_rate,population_65_plus,median_age,hispanic_latino_population,hispanic_latino_rate,total_households,households_with_children,households_with_seniors,avg_household_size,avg_family_size,school_enrolled_population,elementary_school_enrollment,high_school_enrollment,bachelors_or_higher,median_household_income,mean_household_income,poverty_rate,family_poverty_rate,employed_population,unemployed_population,unemployment_rate,drove_alone_count,drove_alone_rate,public_transit_commute_count,public_transit_commute_rate,work_from_home_count,work_from_home_rate,total_housing_units,occupied_housing_units,vacant_housing_units,vacancy_rate,owner_occupied_units,renter_occupied_units,renter_rate,vehicle_households,no_vehicle_households,no_vehicle_rate,one_vehicle_households,two_vehicle_households,three_plus_vehicle_households,median_gross_rent,rent_burden_30_34_count,rent_burden_35_plus_count,tract_type_flag,tract_id,total_population_y,total_crime_count,violent_crime_count,property_crime_count,crime_rate_per_1000,violent_crime_rate_per_1000,property_crime_rate_per_1000,violent_safety_score,property_safety_score,safety_score,low_population_flag,walk_block_group_count,walk_tot_pop,walk_households,walk_housing_units,walk_workers,walk_land_acres,nat_walk_index,d2a_jobs_housing_mix,d2b_employment_mix,d3b_intersection_density,d4a_commute_mode_split,d2a_ranked,d2b_ranked,...,heat_wave_expected_annual_loss_rating,heat_wave_expected_annual_loss_rate_building,heat_wave_expected_annual_loss_rate_national_percentile,heat_wave_hazard_type_risk_index_score,heat_wave_hazard_type_risk_index_rating,landslide_annualized_frequency,landslide_exposure_building_value,landslide_exposure_population,landslide_expected_annual_loss_building_value,landslide_expected_annual_loss_total,landslide_expected_annual_loss_score,landslide_expected_annual_loss_rating,landslide_expected_annual_loss_rate_building,landslide_expected_annual_loss_rate_national_percentile,landslide_hazard_type_risk_index_score,landslide_hazard_type_risk_index_rating,inland_flooding_annualized_frequency,inland_flooding_exposure_building_value,inland_flooding_exposure_population,inland_flooding_expected_annual_loss_building_value,inland_flooding_expected_annual_loss_total,inland_flooding_expected_annual_loss_score,inland_flooding_expected_annual_loss_rating,inland_flooding_expected_annual_loss_rate_building,inland_flooding_expected_annual_loss_rate_national_percentile,inland_flooding_hazard_type_risk_index_score,inland_flooding_hazard_type_risk_index_rating,tsunami_annualized_frequency,tsunami_exposure_building_value,tsunami_exposure_population,tsunami_expected_annual_loss_building_value,tsunami_expected_annual_loss_total,tsunami_expected_annual_loss_score,tsunami_expected_annual_loss_rating,tsunami_expected_annual_loss_rate_building,tsunami_expected_annual_loss_rate_national_percentile,tsunami_hazard_type_risk_index_score,tsunami_hazard_type_risk_index_rating,wildfire_annualized_frequency,wildfire_exposure_building_value,wildfire_exposure_population,wildfire_expected_annual_loss_building_value,wildfire_expected_annual_loss_total,wildfire_expected_annual_loss_score,wildfire_expected_annual_loss_rating,wildfire_expected_annual_loss_rate_building,wildfire_expected_annual_loss_rate_national_percentile,wildfire_hazard_type_risk_index_score,wildfire_hazard_type_risk_index_rating,NAMELSAD_y,school_count,elementary_school_count,middle_school_count,high_school_count,other_school_count,charter_school_count,tract_area_sq_mile_y,schools_per_sq_mile,district_name,district_type,district_grade_low,district_grade_high,district_overlap_pct,ela_students_tested,ela_current_status,ela_status_level,ela_change,ela_change_level,math_students_tested,math_current_status,math_status_level,math_change,math_change_level,academic_strength_score,academic_strength_tier
0,1400000US06073000100,Census Tract 1; San Diego

In [20]:
# checking all columns in the merged dataset
master.columns.tolist()

['geo_id',
 'tract_name',
 'total_population_x',
 'population_under_5',
 'population_under_5_rate',
 'population_under_18',
 'population_under_18_rate',
 'population_65_plus',
 'median_age',
 'hispanic_latino_population',
 'hispanic_latino_rate',
 'total_households',
 'households_with_children',
 'households_with_seniors',
 'avg_household_size',
 'avg_family_size',
 'school_enrolled_population',
 'elementary_school_enrollment',
 'high_school_enrollment',
 'bachelors_or_higher',
 'median_household_income',
 'mean_household_income',
 'poverty_rate',
 'family_poverty_rate',
 'employed_population',
 'unemployed_population',
 'unemployment_rate',
 'drove_alone_count',
 'drove_alone_rate',
 'public_transit_commute_count',
 'public_transit_commute_rate',
 'work_from_home_count',
 'work_from_home_rate',
 'total_housing_units',
 'occupied_housing_units',
 'vacant_housing_units',
 'vacancy_rate',
 'owner_occupied_units',
 'renter_occupied_units',
 'renter_rate',
 'vehicle_households',
 'no_vehic

In [21]:
# saving the merged dataset without filling missing values
master.to_csv(paths['output'], index=False)

print(f'saved to: {paths["output"]}')

saved to: C:\Users\cococ\Desktop\Data Science Projects\capstone-3\data\processed\master_tract_features.csv


In [22]:
# columns used directly for scoring
scoring_groups = {
    'safety': [
        'safety_score',
        'violent_safety_score',
        'property_safety_score'
    ],

    'walkability_access': [
        'nat_walk_index',
        'd2a_ranked',
        'd2b_ranked',
        'd3b_ranked',
        'd4a_ranked',
        'transit_stops_per_sq_mile',
        'has_transit_access',
        'public_transit_commute_rate',
        'no_vehicle_rate'
    ],

    'environment_resilience': [
        'expected_annual_loss_score_composite',
        'social_vulnerability_score',
        'community_resilience_score',
        'heat_wave_hazard_type_risk_index_score',
        'inland_flooding_hazard_type_risk_index_score',
        'wildfire_hazard_type_risk_index_score'
    ],

    'schools': [
        'schools_per_sq_mile',
        'academic_strength_score'
    ]
}

In [23]:
# columns kept for EDA, context, filtering, and interpretation
eda_context_groups = {
    'tract_identity': [
        'tract_id',
        'tract_name',
        'total_population_x',
        'tract_type_flag'
    ],

    'demographic_context': [
        'population_under_5_rate',
        'population_under_18_rate',
        'median_age',
        'hispanic_latino_rate',
        'total_households',
        'households_with_children',
        'avg_household_size',
        'avg_family_size',
        'bachelors_or_higher'
    ],

    'economic_housing_context': [
        'median_household_income',
        'poverty_rate',
        'family_poverty_rate',
        'unemployment_rate',
        'drove_alone_rate',
        'work_from_home_rate',
        'renter_rate',
        'median_gross_rent',
        'rent_burden_30_34_count',
        'rent_burden_35_plus_count',
        'vacancy_rate'
    ],

    'crime_context': [
        'total_crime_count',
        'violent_crime_count',
        'property_crime_count',
        'crime_rate_per_1000',
        'violent_crime_rate_per_1000',
        'property_crime_rate_per_1000',
        'low_population_flag'
    ],

    'transit_context': [
        'transit_stop_count',
        'large_tract_flag'
    ],

    'environment_context': [
        'expected_annual_loss_rating_composite',
        'social_vulnerability_rating',
        'community_resilience_rating',
        'heat_wave_hazard_type_risk_index_rating',
        'inland_flooding_hazard_type_risk_index_rating',
        'wildfire_hazard_type_risk_index_rating'
    ],

    'school_context': [
        'school_count',
        'elementary_school_count',
        'middle_school_count',
        'high_school_count',
        'charter_school_count',
        'district_name',
        'district_type',
        'district_grade_low',
        'district_grade_high',
        'district_overlap_pct',
        'academic_strength_tier'
    ]
}

In [24]:
# flattening the groups so they can be used in the dataframe
scoring_cols = [
    col
    for group in scoring_groups.values()
    for col in group
]

eda_context_cols = [
    col
    for group in eda_context_groups.values()
    for col in group
]

mvp_keep_cols = scoring_cols + eda_context_cols

In [25]:
# keeping only columns that actually exist in the merged dataset
mvp_keep_cols = [col for col in mvp_keep_cols if col in master.columns]

mvp_features = master[mvp_keep_cols].copy()

mvp_features.shape

(737, 70)

In [26]:
# columns removed from the MVP dataset
removed_cols = [
    col for col in master.columns
    if col not in mvp_keep_cols
]

removed_cols

['geo_id',
 'population_under_5',
 'population_under_18',
 'population_65_plus',
 'hispanic_latino_population',
 'households_with_seniors',
 'school_enrolled_population',
 'elementary_school_enrollment',
 'high_school_enrollment',
 'mean_household_income',
 'employed_population',
 'unemployed_population',
 'drove_alone_count',
 'public_transit_commute_count',
 'work_from_home_count',
 'total_housing_units',
 'occupied_housing_units',
 'vacant_housing_units',
 'owner_occupied_units',
 'renter_occupied_units',
 'vehicle_households',
 'no_vehicle_households',
 'one_vehicle_households',
 'two_vehicle_households',
 'three_plus_vehicle_households',
 'total_population_y',
 'walk_block_group_count',
 'walk_tot_pop',
 'walk_households',
 'walk_housing_units',
 'walk_workers',
 'walk_land_acres',
 'd2a_jobs_housing_mix',
 'd2b_employment_mix',
 'd3b_intersection_density',
 'd4a_commute_mode_split',
 'NAMELSAD_x',
 'tract_area_sq_mile_x',
 'state_name',
 'state_name_abbreviation',
 'county_name',

In [27]:
rename_cols = {
    'total_population_x': 'total_population',
    'nat_walk_index': 'walkability_index',
    'd2a_ranked': 'jobs_housing_mix_score',
    'd2b_ranked': 'employment_mix_score',
    'd3b_ranked': 'intersection_density_score',
    'd4a_ranked': 'commute_mode_diversity_score',
    'transit_stops_per_sq_mile': 'transit_stop_density',
    'expected_annual_loss_score_composite': 'climate_loss_risk_score',
    'social_vulnerability_score': 'social_vulnerability_score',
    'community_resilience_score': 'community_resilience_score',
    'heat_wave_hazard_type_risk_index_score': 'heat_risk_score',
    'inland_flooding_hazard_type_risk_index_score': 'flood_risk_score',
    'wildfire_hazard_type_risk_index_score': 'wildfire_risk_score',
    'schools_per_sq_mile': 'school_density',
    'academic_strength_score': 'school_academic_score'}

mvp_features = mvp_features.rename(columns=rename_cols)

## Cleaning irrelevant features

From 215 to 70 features.

The full merged dataset had over 200 columns, so I created a smaller MVP dataset for EDA and scoring.

I kept columns that were the most straight forward for safety, walkability, transit access, environmental risk, school access, and basic neighborhood context. For scoring I kept rates over counts, since some tracts are larger or more populated than others. For example, `transit_stops_per_sq_mile` is more useful than just `transit_stop_count` because larger tracts can naturally have more stops.

I kept some demographic, income, housing, and school district fields for EDA/context, so I have some idea of what kind of neighborhood each tract represents, but they might not play into the final scoring system.

**Removed columns:** most raw count fields, duplicate merge columns, and very detailed hazard fields. If I were doing a larger scope or a deep analysis, I'd keep them, but for this version, I’m using a smaller set of clear features.

**Renaming:**
Some of the column names were unclear, so I renamed them to something easier to interpret.

In [31]:
master_final = mvp_features.copy()

master_final.to_csv(paths['output'], index=False)

In [32]:
# saving the smaller final dataset for EDA and scoring
master_final_path = processed_path / 'master_final_tract_features.csv'

master_final.to_csv(master_final_path, index=False)

print(f'saved to: {master_final_path}')

saved to: C:\Users\cococ\Desktop\Data Science Projects\capstone-3\data\processed\master_final_tract_features.csv


In [33]:
# renaming the smaller working dataframe
master_final = mvp_features.copy()

master_final.shape

(737, 70)

In [37]:
# checking missing values in the smaller final dataset
master_final_missing = (
    master_final.isna()
    .sum()
    .reset_index()
    .rename(columns={'index': 'column', 0: 'missing_count'}))

master_final_missing['missing_pct'] = (
    master_final_missing['missing_count'] / len(master_final) * 100).round(2)

master_final_missing = master_final_missing[master_final_missing['missing_count'] > 0]

master_final_missing.sort_values('missing_count', ascending=False)

,column,missing_count,missing_pct
3,walkability_index,201,27.27
6,intersection_density_score,201,27.27
5,employment_mix_score,201,27.27
4,jobs_housing_mix_score,201,27.27
7,commute_mode_diversity_score,201,27.27
40,median_gross_rent,87,11.80
33,median_household_income,16,2.17
31,avg_family_size,6,0.81
35,family_poverty_rate,6,0.81
30,avg_household_size,6,0.81


There are 5 features with more than 25% missing values, so I want to investigate that first and see if something went wrong in the merge or which tracts they represent.

In [38]:
# looking at tracts missing walkability data
missing_walkability = master_final[master_final['walkability_index'].isna()].copy()

missing_walkability.shape

(201, 70)

In [41]:
# previewing missing walkability tracts
missing_walkability[
    [
        'tract_id',
        'tract_name',
        'total_population',
        'tract_type_flag',
        'median_household_income',
        'poverty_rate',
        'safety_score',
        'transit_stop_density',
        'has_transit_access']].head(25)

,tract_id,tract_name,total_population,tract_type_flag,median_household_income,poverty_rate,safety_score,transit_stop_density,has_transit_access
3,06073000301,Census Tract 3.01; San Diego County; California,2311,residential_or_mixed,87813.0,15.4,21.002729,45.122430,1
4,06073000302,Census Tract 3.02; San Diego County; California,2873,residential_or_mixed,89573.0,9.4,7.175989,52.115092,1
10,06073000901,Census Tract 9.01; San Diego County; California,3632,residential_or_mixed,102905.0,13.0,12.005457,44.332789,1
11,06073000902,Census Tract 9.02; San Diego County; California,2441,residential_or_mixed,88229.0,12.7,17.639836,66.861631,1
14,06073001201,Census Tract 12.01; San Diego County; California,1982,residential_or_mixed,68578.0,10.4,11.998636,40.660570,1
15,06073001202,Census Tract 12.02; San Diego County; California,3054,residential_or_mixed,100670.0,17.1,15.354707,14.101867,1
16,06073001301,Census Tract 13.01; San Diego County; California,2816,residential_or_mixed,100764.0,9.3,3.601637,73.543688,1
17,06073001302,Census Tract 13.02; San Diego County; California,3238,residential_or_mixed,77459.0,7.4,14.740791,17.668546,1
22,06073001801,Census Tract 18.01; San Diego County; California,1982,residential_or_mixed,83065.0,9.0,10.129604,24.749224,1
23,06073001802,Census Tract 18.02; San Diego County; California,3501,residential_or_mixed,84885.0,13.7,19.399727,59.369198,1


In [42]:
# comparing tracts with and without walkability data
master_final['missing_walkability_flag'] = master_final['walkability_index'].isna().astype(int)

master_final.groupby('missing_walkability_flag')[
    [
        'total_population',
        'median_household_income',
        'poverty_rate',
        'safety_score',
        'transit_stop_density',
        'has_transit_access',
        'no_vehicle_rate']].median()

,total_population,median_household_income,poverty_rate,safety_score,transit_stop_density,has_transit_access,no_vehicle_rate
missing_walkability_flag,,,,,,,
0,4446.5,105867.0,8.45,56.357435,10.649391,1.0,4.00
1,3819.0,120258.0,6.50,74.580491,4.736561,1.0,3.15


## Missing Values: Walkability

I reviewed the missing walkability values and saw that they came from the processed walkability dataset, not from the final merge.
Walkability data was not available for every tract. The processed walkability file had 628 unique San Diego County tracts, and the master tract dataset includes 737 tracts. Because of that, 201 tracts have missing walkability values after the final merge. It's possible that the tract info changed between 2020 (when the walkability data was gathered) and 2024 (the census dataset I was working with originally). I kept a `missing_walkability_flag` instead of filling these values because missing walkability does not necessarily mean low walkability.

<b>Other approaches I could take:</b>
- geographically map the missing tracts to see if they're suburban, military, rural, coastal, etc. I can impute 0 walkability for certain areas.
-  impute values based on other nearby tracts (this isn't guaranteed to be correct though). I could use the average or median of the 3–5 nearest tracts.
- find a new dataset and see if it has the matching census tracts for complete data
- remove walkability from the exercise


In [53]:
master_final.shape # added flag for missing walkability, should be 1 more than the original df

(737, 71)

## Missing Values: Median Gross Rent

I'll need to check missing rows by population, renter rate, and tract type. Some tracts might have be more owner occupied or be military housing.

## Missing Values: Median Household Income 

I'm going to manually inspect these since they are only 16 missing values to see if I can impute them with the San Diego median, or put them at 0. 



In [54]:
# creating a dataframe of tracts missing median rent
missing_rent = master_final[master_final['median_gross_rent'].isna()].copy()

print(f'missing rent rows: {missing_rent.shape[0]}')
print(f'missing rent columns: {missing_rent.shape[1]}')

missing rent rows: 87
missing rent columns: 71


In [55]:
# previewing tracts missing median rent
missing_rent[
    [
        'tract_id',
        'tract_name',
        'total_population',
        'tract_type_flag',
        'median_household_income',
        'poverty_rate',
        'renter_rate',
        'vacancy_rate',
        'safety_score',
        'transit_stop_density'
    ]
].head(25)

,tract_id,tract_name,total_population,tract_type_flag,median_household_income,poverty_rate,renter_rate,vacancy_rate,safety_score,transit_stop_density
0,06073000100,Census Tract 1; San Diego County; California,2948,residential_or_mixed,231667.0,2.2,9.4,8.9,31.241473,10.115410
90,06073003800,Census Tract 38; San Diego County; California,4573,likely_institutional_or_group_quarters,NaN,0.0,100.0,44.4,36.555252,10.594033
115,06073005500,Census Tract 55; San Diego County; California,290,likely_institutional_or_group_quarters,NaN,0.0,100.0,0.0,NaN,11.762391
124,06073006200,Census Tract 62; San Diego County; California,28,likely_non_residential,NaN,NaN,NaN,NaN,NaN,10.704749
125,06073006300,Census Tract 63; San Diego County; California,2038,likely_institutional_or_group_quarters,NaN,NaN,NaN,NaN,30.409277,1.628799
127,06073006600,Census Tract 66; San Diego County; California,2032,residential_or_mixed,110651.0,6.0,100.0,26.6,16.105048,15.256745
131,06073007002,Census Tract 70.02; San Diego County; California,3025,residential_or_mixed,184167.0,4.6,11.8,3.2,40.040928,10.765830
134,06073007302,Census Tract 73.02; San Diego County; California,2610,residential_or_mixed,183529.0,2.1,20.7,4.6,38.956344,0.000000
158,06073008202,Census Tract 82.02; San Diego County; California,1203,residential_or_mixed,111836.0,3.9,63.7,38.3,8.240109,0.000000
159,06073008301,Census Tract 83.01; San Diego County; California,3190,residential_or_mixed,216823.0,2.4,16.9,0.0,28.710778,0.000000


## Missing Values: Median Gross Rent
- since some tracts have low rental rates, I'm going to set a threshold to allow anything with rental rates <20%, and just flag it as a low rate area.
- for higher rental areas, I'll look at nearby tracts and impute a median

In [56]:
# filtering tracts with low renter rates
low_renter_rate = master_final[master_final['renter_rate'] < 20].copy()

low_renter_rate.shape

(158, 71)

In [60]:
# comparing missing rent by renter rate group
print(f'total missing rent: {master_final["median_gross_rent"].isna().sum()}')
print(f'missing rent with renter_rate below 20%: {missing_rent_low_renter.shape[0]}')

total missing rent: 87
missing rent with renter_rate below 20%: 49


In [63]:
# checking missing rent tracts with low renter rates
missing_rent_low_renter = master_final[
    (master_final['median_gross_rent'].isna()) &
    (master_final['renter_rate'] < 20)
].copy()

missing_rent_low_renter[
    [
        'tract_id',
        'tract_name',
        'total_population',
        'renter_rate',
        'median_gross_rent',
        'median_household_income',
        'tract_type_flag']].sort_values(by='median_household_income', ascending=False).head(10)

,tract_id,tract_name,total_population,renter_rate,median_gross_rent,median_household_income,tract_type_flag
654,06073020034,Census Tract 200.34; San Diego County; California,2549,6.6,NaN,247222.0,residential_or_mixed
509,06073017064,Census Tract 170.64; San Diego County; California,8172,5.6,NaN,243880.0,residential_or_mixed
0,06073000100,Census Tract 1; San Diego County; California,2948,9.4,NaN,231667.0,residential_or_mixed
172,06073008331,Census Tract 83.31; San Diego County; California,2648,9.9,NaN,230956.0,residential_or_mixed
529,06073017306,Census Tract 173.06; San Diego County; California,2670,19.1,NaN,229406.0,residential_or_mixed
490,06073017045,Census Tract 170.45; San Diego County; California,2293,7.6,NaN,228750.0,residential_or_mixed
489,06073017044,Census Tract 170.44; San Diego County; California,5674,14.1,NaN,225882.0,residential_or_mixed
498,06073017053,Census Tract 170.53; San Diego County; California,3462,6.5,NaN,224615.0,residential_or_mixed
512,06073017067,Census Tract 170.67; San Diego County; California,2979,18.5,NaN,224149.0,residential_or_mixed
167,06073008313,Census Tract 83.13; San Diego County; California,2195,13.8,NaN,221536.0,residential_or_mixed


I'm not sure if my final scoring method will depend on rental rates, so I'll leave these values for now. Next I want to inspect missing values in `median_gross_rent` where rental rates are above 20% to see if I can impute values there. 

In [64]:
# checking missing rent where renter rate is 20% or higher
missing_rent_higher_renter = master_final[
    (master_final['median_gross_rent'].isna()) &
    (master_final['renter_rate'] >= 20)].copy()

print(f'missing rent with renter_rate 20% or higher: {missing_rent_higher_renter.shape[0]}')

missing rent with renter_rate 20% or higher: 33


In [65]:
# reviewing tracts where rent is missing but renter rate is meaningful
missing_rent_higher_renter[
    [
        'tract_id',
        'tract_name',
        'total_population',
        'renter_rate',
        'median_gross_rent',
        'median_household_income',
        'poverty_rate',
        'vacancy_rate',
        'tract_type_flag']].sort_values(by='renter_rate', ascending=False)

,tract_id,tract_name,total_population,renter_rate,median_gross_rent,median_household_income,poverty_rate,vacancy_rate,tract_type_flag
90,06073003800,Census Tract 38; San Diego County; California,4573,100.0,NaN,NaN,0.0,44.4,likely_institutional_or_group_quarters
115,06073005500,Census Tract 55; San Diego County; California,290,100.0,NaN,NaN,0.0,0.0,likely_institutional_or_group_quarters
127,06073006600,Census Tract 66; San Diego County; California,2032,100.0,NaN,110651.0,6.0,26.6,residential_or_mixed
248,06073009400,Census Tract 94; San Diego County; California,4394,100.0,NaN,89893.0,9.2,9.2,residential_or_mixed
256,06073009511,Census Tract 95.11; San Diego County; California,4222,98.4,NaN,78102.0,10.2,14.1,residential_or_mixed
255,06073009510,Census Tract 95.10; San Diego County; California,4109,97.9,NaN,60497.0,12.7,21.3,residential_or_mixed
240,06073009201,Census Tract 92.01; San Diego County; California,6288,77.8,NaN,86111.0,5.7,8.8,residential_or_mixed
213,06073008381,Census Tract 83.81; San Diego County; California,3255,68.4,NaN,201319.0,8.7,8.8,residential_or_mixed
158,06073008202,Census Tract 82.02; San Diego County; California,1203,63.7,NaN,111836.0,3.9,38.3,residential_or_mixed
525,06073017201,Census Tract 172.01; San Diego County; California,2169,58.7,NaN,202045.0,2.0,28.0,residential_or_mixed


Tract 38 and 55 are Naval property, so I'll remove those since there's no development opportunity.

In [66]:
# comparing missing rent tracts by renter rate group
print(f'total missing rent: {master_final["median_gross_rent"].isna().sum()}')
print(f'missing rent with renter_rate below 20%: {missing_rent_low_renter.shape[0]}')
print(f'missing rent with renter_rate 20% or higher: {missing_rent_higher_renter.shape[0]}')

total missing rent: 87
missing rent with renter_rate below 20%: 49
missing rent with renter_rate 20% or higher: 33


In [ ]:
# flagging missing rent and low renter rate
master_final['missing_rent_flag'] = master_final['median_gross_rent'].isna().astype(int)
master_final['low_renter_rate_flag'] = (master_final['renter_rate'] < 20).astype(int)

I'm going to flag group quarters / institutional tracts to help clean up the dataset and make it easier to sift through.

In [67]:
# checking tract type counts
master_final['tract_type_flag'].value_counts(dropna=False)

tract_type_flag
residential_or_mixed                      727
likely_institutional_or_group_quarters      7
likely_non_residential                      3
Name: count, dtype: int64

In [72]:
# flagging tracts that don't fit the scoring use case
master_final['exclude_from_scoring_flag'] = (
    master_final['tract_type_flag'].isin([
        'likely_institutional_or_group_quarters',
        'likely_non_residential'])).astype(int)

In [73]:
# checking excluded tract count
master_final['exclude_from_scoring_flag'].value_counts()

exclude_from_scoring_flag
0    727
1     10
Name: count, dtype: int64

In [75]:
# reviewing excluded tracts
master_final[
    master_final['exclude_from_scoring_flag'] == 1
][
    [
        'tract_id',
        'tract_name',
        'total_population',
        'tract_type_flag',
        'renter_rate',
        'median_gross_rent',
        'median_household_income']]

,tract_id,tract_name,total_population,tract_type_flag,renter_rate,median_gross_rent,median_household_income
90,06073003800,Census Tract 38; San Diego County; California,4573,likely_institutional_or_group_quarters,100.0,NaN,NaN
115,06073005500,Census Tract 55; San Diego County; California,290,likely_institutional_or_group_quarters,100.0,NaN,NaN
124,06073006200,Census Tract 62; San Diego County; California,28,likely_non_residential,NaN,NaN,NaN
125,06073006300,Census Tract 63; San Diego County; California,2038,likely_institutional_or_group_quarters,NaN,NaN,NaN
239,06073009109,Census Tract 91.09; San Diego County; California,3281,likely_institutional_or_group_quarters,100.0,2750.0,176250.0
268,06073009901,Census Tract 99.01; San Diego County; California,1310,likely_institutional_or_group_quarters,55.6,NaN,NaN
269,06073009902,Census Tract 99.02; San Diego County; California,0,likely_non_residential,NaN,NaN,NaN
280,06073010016,Census Tract 100.16; San Diego County; California,3170,likely_institutional_or_group_quarters,NaN,NaN,NaN
304,06073011300,Census Tract 113; San Diego County; California,2304,likely_institutional_or_group_quarters,100.0,2883.0,111397.0
736,06073990100,Census Tract 9901; San Diego County; California,0,likely_non_residential,NaN,NaN,NaN


In [76]:
# checking missing rent where renter rate is 20% or higher
# and the tract is not excluded from scoring
missing_rent_above_20 = master_final[
    (master_final['median_gross_rent'].isna()) &
    (master_final['renter_rate'] >= 20) &
    (master_final['exclude_from_scoring_flag'] == 0)].copy()

print(f'missing rent rows above 20% renter rate: {missing_rent_above_20.shape[0]}')

missing rent rows above 20% renter rate: 30


In [77]:
# reviewing remaining missing rent tracts above 20% renter rate
missing_rent_above_20[
    [
        'tract_id',
        'tract_name',
        'total_population',
        'tract_type_flag',
        'renter_rate',
        'median_gross_rent',
        'median_household_income',
        'poverty_rate',
        'vacancy_rate']].sort_values(by='renter_rate', ascending=False)

,tract_id,tract_name,total_population,tract_type_flag,renter_rate,median_gross_rent,median_household_income,poverty_rate,vacancy_rate
127,06073006600,Census Tract 66; San Diego County; California,2032,residential_or_mixed,100.0,NaN,110651.0,6.0,26.6
248,06073009400,Census Tract 94; San Diego County; California,4394,residential_or_mixed,100.0,NaN,89893.0,9.2,9.2
256,06073009511,Census Tract 95.11; San Diego County; California,4222,residential_or_mixed,98.4,NaN,78102.0,10.2,14.1
255,06073009510,Census Tract 95.10; San Diego County; California,4109,residential_or_mixed,97.9,NaN,60497.0,12.7,21.3
240,06073009201,Census Tract 92.01; San Diego County; California,6288,residential_or_mixed,77.8,NaN,86111.0,5.7,8.8
213,06073008381,Census Tract 83.81; San Diego County; California,3255,residential_or_mixed,68.4,NaN,201319.0,8.7,8.8
158,06073008202,Census Tract 82.02; San Diego County; California,1203,residential_or_mixed,63.7,NaN,111836.0,3.9,38.3
525,06073017201,Census Tract 172.01; San Diego County; California,2169,residential_or_mixed,58.7,NaN,202045.0,2.0,28.0
205,06073008373,Census Tract 83.73; San Diego County; California,3323,residential_or_mixed,46.1,NaN,192031.0,5.2,1.0
204,06073008372,Census Tract 83.72; San Diego County; California,3819,residential_or_mixed,43.5,NaN,NaN,7.1,4.9


In [79]:
# checking missing rent by renter rate and scoring eligibility
print(f'total missing rent: {master_final["median_gross_rent"].isna().sum()}')

print(
    'missing rent, renter_rate below 20%:',
    master_final[
        (master_final['median_gross_rent'].isna()) &
        (master_final['renter_rate'] < 20)].shape[0])

print(
    'missing rent, renter_rate 20% or higher, not excluded:',
    missing_rent_above_20.shape[0])

print(
    'missing rent, excluded from scoring:',
    master_final[
        (master_final['median_gross_rent'].isna()) &
        (master_final['exclude_from_scoring_flag'] == 1)].shape[0])

total missing rent: 87
missing rent, renter_rate below 20%: 49
missing rent, renter_rate 20% or higher, not excluded: 30
missing rent, excluded from scoring: 8


Ultimately, there are 30 null values that I'd want to expect, but since I'm not 100% sure on how I'm going to score each tract yet, I'm going to leave it null for now and move onto the next set of null values. 

## Missing Values: Median Household Income 

In [80]:
# checking missing median household income
missing_income = master_final[
    master_final['median_household_income'].isna()
].copy()

print(f'missing median household income rows: {missing_income.shape[0]}')

missing median household income rows: 16


In [82]:
# reviewing tracts with missing median household income
missing_income[
    [
        'tract_id',
        'tract_name',
        'total_population',
        'tract_type_flag',
        'median_household_income',
        'poverty_rate',
        'family_poverty_rate',
        'renter_rate',
        'median_gross_rent',
        'vacancy_rate',
        'exclude_from_scoring_flag']].sort_values(by='total_population')

,tract_id,tract_name,total_population,tract_type_flag,median_household_income,poverty_rate,family_poverty_rate,renter_rate,median_gross_rent,vacancy_rate,exclude_from_scoring_flag
736,06073990100,Census Tract 9901; San Diego County; California,0,likely_non_residential,NaN,NaN,NaN,NaN,NaN,NaN,1
269,06073009902,Census Tract 99.02; San Diego County; California,0,likely_non_residential,NaN,NaN,NaN,NaN,NaN,NaN,1
124,06073006200,Census Tract 62; San Diego County; California,28,likely_non_residential,NaN,NaN,NaN,NaN,NaN,NaN,1
115,06073005500,Census Tract 55; San Diego County; California,290,likely_institutional_or_group_quarters,NaN,0.0,0.0,100.0,NaN,0.0,1
268,06073009901,Census Tract 99.01; San Diego County; California,1310,likely_institutional_or_group_quarters,NaN,0.0,0.0,55.6,NaN,0.0,1
125,06073006300,Census Tract 63; San Diego County; California,2038,likely_institutional_or_group_quarters,NaN,NaN,NaN,NaN,NaN,NaN,1
523,06073017112,Census Tract 171.12; San Diego County; California,2319,residential_or_mixed,NaN,1.7,1.3,2.3,NaN,6.7,0
165,06073008311,Census Tract 83.11; San Diego County; California,2739,residential_or_mixed,NaN,3.3,0.9,4.7,NaN,9.0,0
280,06073010016,Census Tract 100.16; San Diego County; California,3170,likely_institutional_or_group_quarters,NaN,NaN,NaN,NaN,NaN,NaN,1
204,06073008372,Census Tract 83.72; San Diego County; California,3819,residential_or_mixed,NaN,7.1,6.4,43.5,NaN,4.9,0


In [84]:
# checking missing median household income for residential tracts only
missing_income_residential = master_final[
    (master_final['median_household_income'].isna()) &
    (master_final['tract_type_flag'] == 'residential_or_mixed')
].copy()

print(f'missing income residential rows: {missing_income_residential.shape[0]}')

missing income residential rows: 8


In [86]:
# reviewing residential tracts with missing income
missing_income_residential[
    [
        'tract_id',
        'tract_name',
        'total_population',
        'tract_type_flag',
        'median_household_income',
        'poverty_rate',
        'family_poverty_rate',
        'renter_rate',
        'median_gross_rent',
        'vacancy_rate']].sort_values(by='total_population')

,tract_id,tract_name,total_population,tract_type_flag,median_household_income,poverty_rate,family_poverty_rate,renter_rate,median_gross_rent,vacancy_rate
523,06073017112,Census Tract 171.12; San Diego County; California,2319,residential_or_mixed,NaN,1.7,1.3,2.3,NaN,6.7
165,06073008311,Census Tract 83.11; San Diego County; California,2739,residential_or_mixed,NaN,3.3,0.9,4.7,NaN,9.0
204,06073008372,Census Tract 83.72; San Diego County; California,3819,residential_or_mixed,NaN,7.1,6.4,43.5,NaN,4.9
728,06073021501,Census Tract 215.01; San Diego County; California,3844,residential_or_mixed,NaN,2.2,1.5,19.3,NaN,5.0
206,06073008374,Census Tract 83.74; San Diego County; California,3952,residential_or_mixed,NaN,1.6,1.5,11.0,NaN,0.0
507,06073017062,Census Tract 170.62; San Diego County; California,4767,residential_or_mixed,NaN,1.4,0.7,7.7,NaN,18.7
250,06073009504,Census Tract 95.04; San Diego County; California,6428,residential_or_mixed,NaN,3.2,4.0,21.4,NaN,10.0
170,06073008328,Census Tract 83.28; San Diego County; California,7814,residential_or_mixed,NaN,4.9,4.7,10.4,NaN,13.6


In [88]:
# keeping missing median income values for now
# these will be reviewed again before scoring
missing_income_residential.shape

(8, 72)

There were 8  mixed residential tracts missing median household income. I left these values missing for now instead of imputing since I'm not sure how they'll contribue to scoring yet. 

## Remanining Null Values

In [109]:
# checking remaining missing values between 1 and 15
remaining_nulls = master_final.isna().sum().sort_values(ascending=False)

remaining_nulls = remaining_nulls[
    (remaining_nulls < 16) & 
    (remaining_nulls > 0)].reset_index()

remaining_nulls.columns = ['column', 'missing_count']

remaining_nulls['missing_percent'] = (
    remaining_nulls['missing_count'] / len(master_final) * 100).round(2)

remaining_nulls

,column,missing_count,missing_percent
0,avg_household_size,6,0.81
1,family_poverty_rate,6,0.81
2,avg_family_size,6,0.81
3,poverty_rate,5,0.68
4,no_vehicle_rate,5,0.68
5,vacancy_rate,5,0.68
6,renter_rate,5,0.68
7,property_safety_score,4,0.54
8,safety_score,4,0.54
9,violent_safety_score,4,0.54


In [103]:
# filtering to tracts that are relevant for scoring
master_scoring_check = master_final[
    master_final['exclude_from_scoring_flag'] == 0
].copy()

master_scoring_check.shape

(727, 72)

In [112]:
# getting the remaining null columns from the summary table
small_null_cols = remaining_nulls['column'].tolist()

# finding rows that have at least one of these small null values
small_null_rows = master_final[
    master_final[small_null_cols].isna().any(axis=1)
].copy()

# checking tract types for rows with small missing values
small_null_rows['tract_type_flag'].value_counts(dropna=False)

tract_type_flag
likely_institutional_or_group_quarters    4
likely_non_residential                    3
Name: count, dtype: int64

In [122]:
# filtering to tracts that fit the scoring use case
master_residential = master_final[
    master_final['tract_type_flag'] == 'residential_or_mixed'].copy()

print(f'master_final rows: {master_final.shape[0]}')
print(f'master_residential rows: {master_residential.shape[0]}')

master_final rows: 737
master_residential rows: 727


In [121]:
# checking missing values for residential/mixed tracts only
residential_nulls = master_residential.isna().sum().sort_values(ascending=False)

residential_nulls = residential_nulls[
    residential_nulls > 0].reset_index()

residential_nulls.columns = ['column', 'missing_count']

residential_nulls['missing_percent'] = (
    residential_nulls['missing_count'] / len(master_residential) * 100).round(2)

residential_nulls

,column,missing_count,missing_percent
0,walkability_index,199,27.37
1,intersection_density_score,199,27.37
2,employment_mix_score,199,27.37
3,jobs_housing_mix_score,199,27.37
4,commute_mode_diversity_score,199,27.37
5,median_gross_rent,79,10.87
6,median_household_income,8,1.10


After inspecting further, I found that all of the remaining missing values were not for residential or mixed tracts. I'm going to move forward focusing only on residential tracts which should be easier for scoring and feature engineering. 

`master_residential` is the main dataset I’ll use moving forward. The full merged file is still saved, but this filtered version is a better fit for EDA and scoring because it focuses on tracts that could realistically be evaluated for neighborhood development or community investment.

In [124]:
# saving residential/mixed dataset for future EDA and scoring
master_residential_path = processed_path / 'master_residential_tract_features.csv'

master_residential.to_csv(master_residential_path, index=False)

master_residential_path

WindowsPath('C:/Users/cococ/Desktop/Data Science Projects/capstone-3/data/processed/master_residential_tract_features.csv')

In [125]:
# checking saved file
master_residential_check = pd.read_csv(master_residential_path)

master_residential_check.shape

(727, 72)

In [137]:
# confirming that we only have residential/mixed in this df
master_residential['tract_type_flag'].value_counts()

tract_type_flag
residential_or_mixed    727
Name: count, dtype: int64

## Final Merge Summary

In this notebook, I combined all of the cleaned tract-level datasets into one main dataset.

The previous wrangling notebooks cleaned each data source separately:

- ACS demographic and housing data
- crime and safety data
- transit access data
- walkability data
- environmental risk data
- school access data

Each dataset was cleaned at the census tract level so it could be merged into one master table. I used ACS as the base dataset because it had the full tract list, then merged the other datasets using a standardized `tract_id`.

After merging, I checked the row counts, tract ID formats, duplicate columns, and missing values. I kept the full merged dataset as `master_final`, but narrowed it down to a smaller set of useful features for EDA and scoring. I also created `master_residential`, which keeps only residential or mixed-use tracts. This removes institutional, group quarters, and likely non-residential areas because those tracts don't fit the main development opportunity use case.

Some missing values are still left in the dataset. I decided not to fill them during this notebook because I don't know yet which features will be part of the final scoring method. For things like walkability or median income, I may average nearby tracts then impute the null values. 

Moving forward, I’ll use `master_residential` for EDA and scoring. The next step is to do EDA: further explore the features, understand patterns across tracts, and decide which variables should go into the final neighborhood opportunity score.